In [ ]:
import dt4dds_benchmark
import pandas as pd
import plotly.express as px

In [ ]:
genscript_df = pd.read_csv('./pricing_genscript.csv', sep=',')
genscript_df['price'] = genscript_df['CHF'].astype(float)*1.24 # convert CHF to USD
genscript_df.drop(columns=['CHF'], inplace=True)
genscript_df['supplier'] = 'Genscript'

genscript_df

In [ ]:
twist_df = pd.read_csv('./pricing_twist.csv', sep=',')
twist_df['price'] = twist_df['USD'].astype(float) # no conversion needed
twist_df.drop(columns=['USD'], inplace=True)
twist_df['supplier'] = 'Twist'

twist_df

# Number of sequences vs. Price per base

In [ ]:
costdf = pd.concat([genscript_df, twist_df], ignore_index=True)
costdf['price_per_base'] = costdf['price'] / (150 * costdf['n_seqs']) # 150 bases per sequence

fig = px.line(
    costdf,
    x='n_seqs',
    y='price_per_base',
    log_x=True,
    log_y=True,
    color='supplier',
    range_x=(1e3, 1e7),
    range_y=(0.00002, 0.01),
)

fig.update_traces(line_shape='vh')
fig.update_xaxes(title='Number of sequences', dtick=1)
fig.update_yaxes(title='USD per base', dtick=1)
fig.update_layout(
    width=320,
    height=300,
    margin=dict(l=0, r=10, t=5, b=5),
    showlegend=False,
)
fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
fig.write_image('./figures/pricing_sequence_number.svg')
fig.show()

# save the data
costdf.to_csv('./figures/pricing_sequence_number.csv', index=False)

# Data volume vs. Price per Byte

In [ ]:
datadf = costdf.copy()
datadf['total_bases'] = datadf['n_seqs'] * (150-2*20)  # account for primers
datadf['coderate'] = 1.50
datadf.loc[datadf['supplier'] == 'Genscript', 'coderate'] = 1.00
datadf['data_volume_in_bit'] = datadf['total_bases'] * datadf['coderate']
datadf['data_volume_in_byte'] = datadf['data_volume_in_bit'] / 8
datadf['price_per_bit'] = datadf['price_per_base'] / datadf['coderate']
datadf['price_per_kilobyte'] = datadf['price_per_bit'] * 8 * 1e3  # convert to megabytes

fig = px.line(
    datadf,
    x='data_volume_in_byte',
    y='price_per_kilobyte',
    log_x=True,
    log_y=True,
    color='supplier',
    range_x=(1e3, 2e8),
    range_y=(0.2, 500),
)

fig.update_traces(line_shape='vh')
fig.update_xaxes(title='Data volume in Byte', dtick=1)
fig.update_yaxes(title='USD per Kilobyte', dtick=1)
fig.add_vline(x=1e6, line_dash='dash', line_color='black', line_width=2)
fig.update_layout(
    width=320,
    height=300,
    margin=dict(l=0, r=15, t=5, b=5),
    showlegend=False,
)
fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
fig.write_image('./figures/pricing_data_volume.svg')
fig.show()

# save the data
datadf.to_csv('./figures/pricing_data.csv', index=False)

# Cost per byte per codec and per scenario

considering storage of one megabyte at varying storage densities, at a fixed sequencing depth of 30x

In [ ]:
codecdf = pd.DataFrame({
    'codec': ['DNA-Aeon', 'DNA-Aeon', 'DNA-Aeon', 'DNA Fountain', 'DNA Fountain', 'DNA Fountain', 'DNA-RS', 'DNA-RS', 'DNA-RS', 'Goldman', 'HEDGES', 'HEDGES'],
    'coderate': [1.5, 1.0, 0.5, 1.5, 1.0, 0.5, 1.5, 1.0, 0.5, 0.34, 1.07, 0.63],
    'twist': [4.1, 1.6, 0.48, 20, 7.6, 24, 2.6, 0.97, 0.66, 18, 3.4, 2.9],
    'genscript': [None, 6.5, 11, None, None, None, None, 11, 9.5, None, 12, 9.5]
})

codecdf = codecdf.melt(id_vars=['codec', 'coderate'], var_name='supplier', value_name='minphys')
codecdf['minphys'] = codecdf['minphys'].astype(float)
codecdf['maxdensity'] = 113.7*codecdf['coderate']/codecdf['minphys']
codecdf['n_bases'] = 8e6 / codecdf['coderate'] # one megabyte
codecdf['n_seqs'] = codecdf['n_bases'] / (150 - 2*20) # 150 bases per sequence, accounting for primers
codecdf['price'] = codecdf.apply(lambda row: twist_df.loc[twist_df['n_seqs'] >= row['n_seqs'], 'price'].min() if row['supplier'] == 'twist' else genscript_df.loc[genscript_df['n_seqs'] >= row['n_seqs'], 'price'].min(), axis=1)
codecdf['price_per_kilobyte'] = codecdf['price'] / 1000

codecdf

In [ ]:
plotdf = codecdf.copy()
plotdf['coderate'] = plotdf['coderate'].map('{:.2f}'.format)

fig = dt4dds_benchmark.analysis.plotting.tiered_bar(
    plotdf,
    "codec",
    "coderate",
    "price_per_kilobyte",
    color_by = "supplier",
)
fig.update_yaxes(
    title_text='USD per Kilobyte',
)
fig.update_layout(
    width=680,
    height=200,
    margin=dict(l=0, r=2, t=2, b=30),
    showlegend=False,
)


fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
fig.update_xaxes(
    tickfont_size=28/3, 
    tickangle=0,
)
fig.write_image('./figures/price_by_codec.svg')
fig.show()

# save the data
plotdf.to_csv('./figures/price_by_codec.csv', index=False)

In [ ]:
plotdf.groupby('supplier')['price_per_kilobyte'].mean()